# CHO WGS 全量分析：物种/株系确认 + Transgene 检测

## 已有结果（5M reads 抽样）

| 项目 | 结果 |
| :--- | :---: |
| 比对率 | 99.85% → 确认为中国仓鼠 (*Cricetulus griseus*) |
| DHFR 状态 | 深度比值正常 → DHFR 完整 |
| 初步株系判断 | CHO-K1 或 CHO-S（全量数据可精确确认）|
| 参考基因组 | GCF_003668045.1 CriGri-PICR |
| 原始数据量 | R1: 17 GB, R2: 16 GB（约 230M read pairs，估算 ~30x 覆盖度）|

---

## 原方案评判

### 正确之处 ✅
- 核心算法"宿主减除法"方向正确，是业界标准做法
- 正确警告不能对全基因组原始数据从头组装（需 500 GB+ 内存）
- 工具选择合理：MEGAHIT 省内存，NCBI remote BLAST 省磁盘

### 需要修正或补充 ❌

1. **必须分步执行，不能用三段管道**：`fastp | bwa | samtools` 三段管道中，fastp 的 HTML/JSON 只在 fastp 退出时写出——若 bwa 或 samtools 崩溃，客户 QC 文件全部丢失。分步执行：Step 1 完成后 QC 文件立即落盘，后续步骤崩溃不影响已有产出。（本服务器三段管道已导致多次 futex 死锁和 OOM 崩溃。）

2. **Transgene 检测必须基于全量 BAM**：5M 抽样中 unmapped reads 约 10K 条，组装效果差；全量数据有约 345K unmapped reads（按 0.15% 未比对率），灵敏度提升 ~34 倍。

3. **Unmapped reads 提取逻辑**：分两类——`-f 12`（双端均未比对）→ PE 输入；`-f 4 -F 8`（一端未比对）→ SE 补充；BAM 先 name-sort 再 fastq。

4. **Contig 过滤**：`--min-contig-len 200`，BLAST 前同样过滤 ≥200 bp。

## 执行计划

| 任务 | 内容 | 估计耗时 | tmux session |
| :--- | :--- | :---: | :---: |
| Step 1 | fastp：QC + 输出 filtered R1/R2 到磁盘 | ~45 min | `cho_wgs` |
| Step 2 | bwa mem -t 20 \| samtools sort（两段管道）| ~2–3 h | `cho_wgs` |
| Step 3 | samtools index | ~10 min | `cho_wgs` |
| Task 1B | flagstat + DHFR 深度 + multiqc | <30 min | 前台运行 |
| Task 2A | 提取 unmapped reads（PE + singleton）| ~30 min | `cho_unmapped` |
| Task 2B | MEGAHIT 从头组装 | ~30–60 min | `cho_megahit` |
| Task 2C | 过滤 ≥200 bp contigs + BLASTn remote | ~10–30 min | 前台运行 |

**磁盘预算**：filtered FASTQ ~30 GB + 全量 BAM ~100–120 GB + unmapped/组装 <5 GB；服务器 5.9 TB 可用，充足。

**日志**：`tail -f /home/gao/projects_2026H2/3_cho_wgs_species_confirm/align/cho_wgs_3step.log`

## Task 1: 全量 fastp QC + BWA 比对

### 策略：三步独立顺序执行（非管道）

```
Step 1: fastp  →  filtered_R1.fq.gz + filtered_R2.fq.gz + QC HTML/JSON（写入磁盘）
Step 2: bwa mem | samtools sort  →  wt1_cho_full.bam（仅两段管道）
Step 3: samtools index  →  wt1_cho_full.bam.bai
```

**为什么不用三段管道（fastp | bwa | samtools）：**
- fastp 的 HTML/JSON 报告只在 fastp **退出时**才写出——三段管道中若 bwa 或 samtools 崩溃，客户得不到任何 QC 文件
- 三段管道同时运行三个工具，内存需求无法精确预测
- 三段管道复杂的进程间阻塞关系是触发 bwa 内部 futex 死锁的根本原因（本服务器已复现两次）
- 分步执行：Step 1 崩溃只损失 QC 时间；Step 2 崩溃不影响已有 QC 文件，也不影响 filtered FASTQ（已在磁盘）

**各步骤线程分配（分步执行，资源可精确预测，可适当提高线程数）：**

| 步骤 | 工具 | 参数 | 线程数 | 内存 |
| :--- | :---: | :---: | :---: | :---: |
| Step 1 | fastp | `-w 8` | 8 | 低 |
| Step 2 | bwa mem | `-t 20` | 20 | ~5G |
| Step 2 | samtools sort | `-@ 8 -m 8G` | 8 | 9×8G=72G 总量 |
| Step 3 | samtools index | `-@ 8` | 8 | 低 |

> `samtools sort -m` 是 **per-thread** 内存（非总量）：`-@ 8 -m 8G` = 9 线程 × 8G = 72G 总量，服务器 125G RAM 可用 102G，充足。

In [ ]:
#!/bin/bash
# ============================================================
# Task 1A: 全量比对 — 三步独立顺序执行（tmux cho_wgs）
# ============================================================
PROJDIR="/home/gao/projects_2026H2/3_cho_wgs_species_confirm"
R1="/home/gao/Dropbox/Keqiang/23TF7FLT4_3_0469165296_wt1_S8_L003_R1_001.fastq.gz"
R2="/home/gao/Dropbox/Keqiang/23TF7FLT4_3_0469165296_wt1_S8_L003_R2_001.fastq.gz"
CHO_REF="/Work_bio/references/Cricetulus_griseus/CriGri-PICR/ncbi_refseq/GCF_003668045.1_CriGri-PICR_genomic.fna"
FILT_R1="$PROJDIR/align/filtered_R1.fq.gz"
FILT_R2="$PROJDIR/align/filtered_R2.fq.gz"
BAM="$PROJDIR/align/wt1_cho_full.bam"
LOG="$PROJDIR/align/cho_wgs_3step.log"

mkdir -p $PROJDIR/{qc,align,results,transgene}

tmux new-session -d -s cho_wgs "conda run -n regular_bioinfo bash -c '
set -euo pipefail

# ── STEP 1: fastp（独立运行，完成后 QC 文件立即写入磁盘）─────────────
echo \"[STEP1 START] \$(date)\" | tee $LOG
fastp \
  -i $R1 -I $R2 \
  -o $FILT_R1 -O $FILT_R2 \
  -j $PROJDIR/qc/wt1_full_fastp.json \
  -h $PROJDIR/qc/wt1_full_fastp.html \
  -w 8 \
  2>&1 | tee -a $LOG
echo \"[STEP1 DONE — QC 文件已就绪] \$(date)\" | tee -a $LOG

# ── STEP 2: bwa mem | samtools sort（仅两段管道）────────────────────────
# bwa stderr 写入 log，stdout SAM 直接传给 samtools
echo \"[STEP2 START] \$(date)\" | tee -a $LOG
bwa mem -t 20 \
  -R \"@RG\tID:wt1\tSM:wt1\tPL:ILLUMINA\" \
  $CHO_REF $FILT_R1 $FILT_R2 \
  2>>$LOG | \
samtools sort -@ 8 -m 8G \
  -o $BAM - \
  2>>$LOG
echo \"[STEP2 DONE] \$(date)\" | tee -a $LOG

# ── STEP 3: samtools index ──────────────────────────────────────────────
echo \"[STEP3 START] \$(date)\" | tee -a $LOG
samtools index -@ 8 $BAM
echo \"[STEP3 DONE — BAM+BAI 完成] \$(date)\" | tee -a $LOG
' 2>&1 | tee -a $LOG"

echo "Pipeline launched in tmux session 'cho_wgs'"
echo "Monitor : tmux capture-pane -t cho_wgs -p"
echo "Log     : tail -f $LOG"

In [ ]:
# Task 1A 进度监控（随时重新运行此 cell）
PROJDIR="/home/gao/projects_2026H2/3_cho_wgs_species_confirm"
LOG="$PROJDIR/align/cho_wgs_3step.log"

echo "=== 当前进程 ==="
ps aux | grep -E "fastp|bwa mem|samtools" | grep -v grep | \
  awk '{printf "PID=%-8s CPU=%-5s %s\n",$2,$3,$11}' || echo "(无运行中进程)"

echo ""
echo "=== 步骤进度 ==="
grep -E "\[STEP[123]" "$LOG" 2>/dev/null || echo "(日志尚无步骤标记)"

echo ""
echo "=== 日志最后 5 行 ==="
tail -5 "$LOG" 2>/dev/null || echo "(日志尚未生成)"

echo ""
echo "=== 输出文件状态 ==="
ls -lh "$PROJDIR/align/filtered_R1.fq.gz" "$PROJDIR/align/filtered_R2.fq.gz" 2>/dev/null || echo "filtered FASTQ: 未完成"
ls -lh "$PROJDIR/qc/wt1_full_fastp.html" "$PROJDIR/qc/wt1_full_fastp.json" 2>/dev/null || echo "QC 文件: 未完成"
ls -lh "$PROJDIR/align/wt1_cho_full.bam" "$PROJDIR/align/wt1_cho_full.bam.bai" 2>/dev/null || echo "BAM/BAI: 未完成"

In [ ]:
# ============================================================
# Step 1 完成后立即运行：MultiQC 汇总报告
# fastp 完成即可运行，无需等待 bwa/samtools
# ============================================================
PROJDIR="/home/gao/projects_2026H2/3_cho_wgs_species_confirm"

conda run -n regular_bioinfo multiqc \
  $PROJDIR/qc/ \
  -o $PROJDIR/qc/multiqc_full \
  --force -q

echo "MultiQC 报告: $PROJDIR/qc/multiqc_full/multiqc_report.html"
ls -lh $PROJDIR/qc/multiqc_full/multiqc_report.html

In [ ]:
# ============================================================
# Task 1B: 全量比对率 + DHFR 株系分析（Task 1A 完成后运行）
# ============================================================
PROJDIR="/home/gao/projects_2026H2/3_cho_wgs_species_confirm"
CHO_GFF="/Work_bio/references/Cricetulus_griseus/CriGri-PICR/ncbi_refseq/GCF_003668045.1_CriGri-PICR_genomic.gff.gz"
BAM="${PROJDIR}/align/wt1_cho_full.bam"

echo "=== 全量比对统计 ==="
samtools flagstat ${BAM} | tee ${PROJDIR}/results/cho_full_flagstat.txt

echo ""
echo "=== DHFR 株系分析（全量深度）==="
DHFR_CHR=$(awk '{print $1}' ${PROJDIR}/results/dhfr_gff.txt | head -1)
DHFR_START=$(awk '{print $4}' ${PROJDIR}/results/dhfr_gff.txt | head -1)
DHFR_END=$(awk '{print $5}' ${PROJDIR}/results/dhfr_gff.txt | head -1)

FLANK_START=$((DHFR_START - 500000))
FLANK_END=$((DHFR_END + 500000))
[ ${FLANK_START} -lt 1 ] && FLANK_START=1

DHFR_COV=$(samtools depth -r ${DHFR_CHR}:${DHFR_START}-${DHFR_END} ${BAM} | \
  awk '{s+=$3; n++} END{printf "%.2f", (n>0)?s/n:0}')
FLANK_COV=$(samtools depth -r ${DHFR_CHR}:${FLANK_START}-${FLANK_END} ${BAM} | \
  awk '{s+=$3; n++} END{printf "%.2f", (n>0)?s/n:0}')
DHFR_BASES=$(samtools depth -r ${DHFR_CHR}:${DHFR_START}-${DHFR_END} ${BAM} | wc -l)
DHFR_LEN=$((DHFR_END - DHFR_START))

python3 - <<EOF
dhfr_cov   = float("${DHFR_COV}")
flank_cov  = float("${FLANK_COV}")
dhfr_bases = int("${DHFR_BASES}")
dhfr_len   = int("${DHFR_LEN}")
ratio      = dhfr_cov / max(flank_cov, 0.001)
pct        = dhfr_bases / dhfr_len * 100 if dhfr_len > 0 else 0

print(f"  DHFR 区域深度         : {dhfr_cov:.1f}x")
print(f"  侧翼 ±500kb 深度      : {flank_cov:.1f}x")
print(f"  DHFR 碱基覆盖率       : {pct:.1f}%")
print(f"  深度比值 DHFR/侧翼    : {ratio:.3f}")
print()
if dhfr_cov < 1 and pct < 5:
    verdict = "DHFR 纯合缺失 → CHO-DG44"
elif ratio < 0.3:
    verdict = "DHFR 半合缺失 → CHO-DXB11"
elif ratio > 0.7:
    verdict = f"DHFR 完整 → CHO-K1 或 CHO-S（全量深度 {flank_cov:.0f}x，可信度高）"
else:
    verdict = "结果模糊，请检查 DHFR 坐标是否正确"
print(f"  判断: {verdict}")
EOF

# MultiQC
multiqc ${PROJDIR}/qc/ -o ${PROJDIR}/qc/multiqc_full --force -q
echo ""
echo "MultiQC 全量报告: ${PROJDIR}/qc/multiqc_full/multiqc_report.html"

## Task 2: Transgene（外源基因）检测

### 策略

```
全量 BAM → 提取非宿主 reads → MEGAHIT 从头组装 → 过滤 ≥200 bp → BLAST 鉴定
```

### 两类目标 reads

| 类型 | samtools 参数 | 含义 | 用途 |
|------|--------------|------|------|
| 双端均未比对 | `-f 12 -F 256 -F 2048` | Transgene 内部序列 | PE 输入 MEGAHIT `-1/-2` |
| 仅一端未比对 | `-f 4 -F 8 -F 256 -F 2048` | Transgene/宿主边界 | SE 补充输入 MEGAHIT `-r` |

**注意**：使用 `-F 256 -F 2048` 排除 secondary 和 supplementary alignment，避免重复计数。

### 为什么必须用全量 BAM（而不是 5M 抽样 BAM）

| | 5M 抽样 | 全量 (~230M reads) |
|--|---------|-------------------|
| 估算 unmapped reads | ~10K | ~345K |
| 组装覆盖度 | 极低，难以出 contig | 足以组装几百 bp 以上片段 |
| 灵敏度 | 低 | 高（提升 ~34 倍）|

### MEGAHIT 可用性检查
如果 `regular_bioinfo` 环境没有 megahit，可用 `mag_biobakery` 环境（已确认有 blastn）。

In [ ]:
# ============================================================
# Task 2A: 提取 unmapped reads（基于全量 BAM，约 1 小时）
# ============================================================
PROJDIR="/home/gao/projects_2026H2/3_cho_wgs_species_confirm"
BAM="${PROJDIR}/align/wt1_cho_full.bam"
TRANS="${PROJDIR}/transgene"
mkdir -p ${TRANS}

tmux new-session -d -s cho_unmapped "
set -euo pipefail
LOG=${TRANS}/extraction.log
echo '[START unmapped extraction] \$(date)' | tee \${LOG}

# --- 类型 1: 双端均未比对 → name-sort → 还原为配对 FASTQ ---
echo '[1/2] Extracting both-unmapped pairs...' | tee -a \${LOG}
samtools view -b -f 12 -F 256 -F 2048 ${BAM} | \
  samtools sort -n -@ 4 -m 4G | \
  samtools fastq \
    -1 ${TRANS}/unmapped_R1.fq.gz \
    -2 ${TRANS}/unmapped_R2.fq.gz \
    -0 /dev/null -s /dev/null -
PE_CNT=\$(zcat ${TRANS}/unmapped_R1.fq.gz | awk 'NR%4==1' | wc -l)
echo \"  PE pairs: \${PE_CNT}\" | tee -a \${LOG}

# --- 类型 2: 仅一端未比对（transgene/宿主边界 reads）→ 单端 FASTQ ---
echo '[2/2] Extracting singleton unmapped reads...' | tee -a \${LOG}
samtools view -b -f 4 -F 8 -F 256 -F 2048 ${BAM} | \
  samtools fastq - | gzip > ${TRANS}/unmapped_singleton.fq.gz
SE_CNT=\$(zcat ${TRANS}/unmapped_singleton.fq.gz | awk 'NR%4==1' | wc -l)
echo \"  SE reads: \${SE_CNT}\" | tee -a \${LOG}

echo '[DONE] \$(date)' | tee -a \${LOG}
"

echo "Job launched in tmux session 'cho_unmapped'"
echo "Monitor : tmux capture-pane -t cho_unmapped -p"
echo "Log     : tail -f ${TRANS}/extraction.log"

In [ ]:
# ============================================================
# Task 2B: MEGAHIT 从头组装（Task 2A 完成后运行）
# ============================================================
PROJDIR="/home/gao/projects_2026H2/3_cho_wgs_species_confirm"
TRANS="${PROJDIR}/transgene"

# 检查 megahit 所在环境
MEGAHIT_BIN=$(conda run -n mag_biobakery which megahit 2>/dev/null || which megahit 2>/dev/null || echo "")
if [ -z "${MEGAHIT_BIN}" ]; then
    echo "ERROR: megahit not found. Install with: mamba install -n regular_bioinfo -c bioconda megahit"
    exit 1
fi
echo "megahit found: ${MEGAHIT_BIN}"

# 输入文件统计
echo "PE pairs : $(zcat ${TRANS}/unmapped_R1.fq.gz | awk 'NR%4==1' | wc -l)"
echo "SE reads : $(zcat ${TRANS}/unmapped_singleton.fq.gz | awk 'NR%4==1' | wc -l)"

# MEGAHIT 不允许输出目录已存在
[ -d ${TRANS}/megahit_out ] && rm -rf ${TRANS}/megahit_out

tmux new-session -d -s cho_megahit "
conda run -n mag_biobakery megahit \
  -1 ${TRANS}/unmapped_R1.fq.gz \
  -2 ${TRANS}/unmapped_R2.fq.gz \
  -r ${TRANS}/unmapped_singleton.fq.gz \
  -o ${TRANS}/megahit_out \
  --min-contig-len 200 \
  -t 16 \
  -m 0.5 \
  2>&1 | tee ${TRANS}/megahit.log
echo '[MEGAHIT DONE] \$(date)' | tee -a ${TRANS}/megahit.log
"

echo "MEGAHIT launched in tmux 'cho_megahit'"
echo "Monitor : tmux capture-pane -t cho_megahit -p"
echo "Log     : tail -f ${TRANS}/megahit.log"

In [ ]:
# ============================================================
# Task 2C: Contig 过滤 + BLASTn 鉴定（Task 2B 完成后运行）
# ============================================================
PROJDIR="/home/gao/projects_2026H2/3_cho_wgs_species_confirm"
TRANS="${PROJDIR}/transgene"
CONTIGS="${TRANS}/megahit_out/final.contigs.fa"

# --- Contig 统计 ---
echo "=== 组装结果统计 ==="
awk '/^>/{next} {print length($0)}' ${CONTIGS} | sort -rn | \
  awk 'BEGIN{n=0; s=0}
    {n++; s+=$1; if(n==1)max=$1}
    $1>=1000{c1000++} $1>=500{c500++} $1>=200{c200++}
    END{
      print "Total contigs   : " n
      print "Total bases     : " s
      print "Longest contig  : " max " bp"
      print ">=1000 bp       : " c1000+0
      print ">=500 bp        : " c500+0
      print ">=200 bp        : " c200+0
    }'

# --- 过滤 >= 200 bp 送 BLAST ---
echo ""
echo "=== 过滤 >= 200 bp ==="
awk 'BEGIN{p=0}
  /^>/{header=$0; p=0; next}
  {seq=$0}
  length(seq) >= 200 {print header; print seq; p=1}' \
  ${CONTIGS} > ${TRANS}/contigs_200bp.fa

N_BLAST=$(grep -c "^>" ${TRANS}/contigs_200bp.fa 2>/dev/null || echo 0)
echo "${N_BLAST} contigs (>=200 bp) will be sent to BLAST"

# --- BLASTn remote ---
if [ "${N_BLAST}" -eq 0 ]; then
    echo "No contigs >= 200 bp. Check assembly quality or unmapped read count."
else
    echo ""
    echo "=== BLASTn (remote NCBI nt, may take several minutes) ==="
    conda run -n mag_biobakery \
    blastn \
      -query ${TRANS}/contigs_200bp.fa \
      -db nt \
      -remote \
      -outfmt "6 qseqid sseqid pident length qlen evalue bitscore stitle" \
      -max_target_seqs 5 \
      -evalue 1e-10 \
      -out ${TRANS}/blast_results.txt

    echo ""
    echo "=== BLAST Top Hits ==="
    printf "%-22s %6s %6s/%s  %10s  %s\n" "Contig" "Ident%" "Aln" "Len" "E-value" "Description"
    printf '%0.s-' {1..100}; echo
    sort -k7 -rn ${TRANS}/blast_results.txt | awk -F'\t' '!seen[$1]++' | head -20 | \
      awk -F'\t' '{printf "%-22s %5s%% %6s/%-6s %10s  %s\n", $1,$3,$4,$5,$6,substr($8,1,60)}'
fi

## 结果解读指南

### Task 1 — 物种/株系判断

| 全量比对率 | 解读 |
|-----------|------|
| ≥ 98% | 确认为中国仓鼠 (CHO) |
| 90–98% | 可能含少量外源污染 |
| < 90% | 非 CHO 或严重污染 |

| DHFR/侧翼深度比值 | 株系判断 |
|------------------|---------|
| < 0.05，且覆盖率 < 5% | DHFR 纯合缺失 → **CHO-DG44** |
| 0.05–0.35 | DHFR 半合缺失 → **CHO-DXB11** |
| > 0.70 | DHFR 完整 → **CHO-K1 或 CHO-S** |

### Task 2 — Transgene 结果解读

| BLAST 结果 | 可能解释 |
|-----------|---------|
| 无命中 | ① unmapped reads 均为测序噪音；② transgene 序列与 CHO 参考高度同源（已被比对捕获）|
| 命中 CMV / SV40 / polyA signal | 重组表达载体元件，确认有外源构建体 |
| 命中抗性基因（Puro / Neo / Hygro） | 选择标记，确认有外源 DNA 插入 |
| 命中人/小鼠基因 | 目的基因（GOI），需与客户确认 |
| 命中病毒载体骨架 | 可能残留包装载体片段 |
| Contigs 无命中但序列 ≥200 bp | 可能为客户保密序列，建议提供载体图谱做定向比对 |

### 局限性说明
- 本方法检测的是**游离/插入**的外源 DNA，**不能确定插入位置**（客户已知晓此点）
- 若 transgene 序列与 CHO 参考高度保守（如某些人源化基因），可能被宿主比对步骤"误吸"而无法检出 → 可改用更严格的参考比对（`-Y` flag + 较高 mapping quality 阈值）或直接向客户索取序列做定向 BLAST
- BLAST remote 速度依赖 NCBI 服务器负载，高峰期可能较慢；如 contigs 较多（>50条），建议分批提交或使用 NCBI Web BLAST 界面逐条确认